# Stage 09 / Lecture 64: Structured Outputs via `TypedDict` and Model Capabilities (`model.profile`)

This production reference notebook implements lightweight schema extraction using Python native **`TypedDict`** and **`Annotated`**, alongside programmatic model capability introspection:
1. **`TypedDict` Schema Definition**: Extracting plain Python dictionaries (`dict`) without runtime validation overhead.
2. **Absence of Runtime Validation**: Demonstrating how `TypedDict` differs from Pydantic by allowing unvalidated runtime mutations.
3. **Nested `TypedDict` Structures**: Modeling hierarchical collections (`List[ActorDict]` inside `MovieDetailsDict`).
4. **Model Capability Introspection (`model.profile`)**: Auditing context window limits, modality support, reasoning, and tool-calling capabilities on the base chat model (and verifying why `.profile` fails on wrapped `RunnableSequence` objects).


In [1]:
from typing import List, Dict, Any, Optional
from typing_extensions import TypedDict, Annotated
import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError("GROQ_API_KEY must be configured in .env for live structured output verification.")

model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("Model initialized successfully for TypedDict and capability auditing.")


Model initialized successfully for TypedDict and capability auditing.


### 1. Lightweight Schema Extraction via `TypedDict` and `Annotated`

Unlike Pydantic `BaseModel`, `TypedDict` returns a standard Python `dict` at runtime without validation overhead.


In [2]:
class MovieDict(TypedDict):
    title: Annotated[str, ..., "The exact title of the movie"]
    year: Annotated[int, ..., "The release year of the movie"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie rating out of 10"]


# 1. Verify no runtime validation occurs on direct instantiation (unlike Pydantic)
unvalidated_dict: MovieDict = {"title": 99999, "year": "twenty-twelve", "director": "Whedon", "rating": 8.0}
assert isinstance(unvalidated_dict, dict)
print("Confirmed zero runtime validation overhead for TypedDict instantiation.")

# 2. Bind TypedDict to LLM
model_with_typedict = model.with_structured_output(MovieDict)
avengers_res = model_with_typedict.invoke("Provide details about the movie The Avengers (2012).")

assert isinstance(avengers_res, dict)
assert "avengers" in avengers_res["title"].lower()
assert avengers_res["year"] == 2012
assert "whedon" in avengers_res["director"].lower()

print("TypedDict Structured Output (Avengers):")
for key, val in avengers_res.items():
    print(f"  {key}: {val} ({type(val).__name__})")


Confirmed zero runtime validation overhead for TypedDict instantiation.


TypedDict Structured Output (Avengers):
  director: Joss Whedon (str)
  rating: 8 (int)
  title: The Avengers (str)
  year: 2012 (int)


### 2. Nested `TypedDict` Structures

Hierarchical extraction works identically with nested `TypedDict` classes, returning pure nested Python dictionaries and lists.


In [3]:
class ActorDict(TypedDict):
    name: Annotated[str, ..., "Full name of the actor"]
    role: Annotated[str, ..., "Character name played in the film"]


class MovieDetailsDict(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The release year"]
    cast: Annotated[List[ActorDict], ..., "Top 3 leading actors"]
    genres: Annotated[List[str], ..., "List of genres"]
    budget_millions_usd: Annotated[Optional[float], ..., "Budget expressed in millions USD (e.g. 220.0)"]


nested_typedict_model = model.with_structured_output(MovieDetailsDict, method="json_schema")
nested_res = nested_typedict_model.invoke(
    "Provide details about The Avengers (2012) including top 3 actors, genres, and budget in millions."
)

assert isinstance(nested_res, dict)
assert len(nested_res["cast"]) >= 2
assert isinstance(nested_res["cast"][0], dict)
assert "name" in nested_res["cast"][0] and "role" in nested_res["cast"][0]

print(f"Nested Dict Output -> {nested_res['title']} ({nested_res['year']}) | Budget: ${nested_res['budget_millions_usd']}M")
print("Cast Members:")
for member in nested_res["cast"]:
    print(f"  - {member['name']} as {member['role']}")


Nested Dict Output -> The Avengers (2012) | Budget: $220.0M
Cast Members:
  - Robert Downey Jr. as Tony Stark / Iron Man
  - Chris Evans as Steve Rogers / Captain America
  - Mark Ruffalo as Bruce Banner / Hulk


### 3. Model Capability Introspection (`model.profile`)

Calling `.profile` on a wrapped structured output runnable raises an `AttributeError` because `with_structured_output()` returns a `RunnableSequence`. Calling `.profile` directly on the base `BaseChatModel` inspects token limits, modalities, reasoning, and tool calling capabilities.


In [4]:
# 1. Verify wrapped runnable raises AttributeError on .profile
try:
    _ = model_with_typedict.profile
    raise AssertionError("Expected AttributeError when accessing .profile on RunnableSequence.")
except AttributeError as exc:
    print(f"Verified expected error on wrapped model.profile: {exc}")

# 2. Inspect base ChatModel profile
profile: Dict[str, Any] = model.profile
assert isinstance(profile, dict)
assert profile.get("tool_calling") is True
assert profile.get("structured_output") is True
assert profile.get("max_input_tokens", 0) > 0

print("\nBase Model Capability Profile:")
key_fields = [
    "name", "max_input_tokens", "max_output_tokens",
    "text_inputs", "image_inputs", "audio_inputs",
    "reasoning_output", "tool_calling", "structured_output"
]
for k in key_fields:
    print(f"  {k:20s}: {profile.get(k)}")

print("\nAll Stage 09 / Lecture 64 TypedDict and model.profile assertions passed.")


Verified expected error on wrapped model.profile: 'RunnableSequence' object has no attribute 'profile'

Base Model Capability Profile:
  name                : GPT OSS 20B
  max_input_tokens    : 131072
  max_output_tokens   : 65536
  text_inputs         : True
  image_inputs        : False
  audio_inputs        : False
  reasoning_output    : True
  tool_calling        : True
  structured_output   : True

All Stage 09 / Lecture 64 TypedDict and model.profile assertions passed.
